In [2]:
import logging
import sys
logger = logging.getLogger('cosipy')
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(sys.stdout))
from cosipy.util import fetch_wasabi_file

# dependencies for response
import numpy as np
import astropy.units as u
from astropy.units import Quantity
from astropy.coordinates import SkyCoord, cartesian_to_spherical, Galactic
from astropy.io import fits
from astropy.time import Time
from astropy.table import Table
import matplotlib.pyplot as plt

# dependencies for plotting mollweide projections
from mhealpy import HealpixMap, HealpixBase
import healpy as hp
import pandas as pd
from pathlib import Path

# This is cosi.response
from scoords import Attitude, SpacecraftFrame
from cosipy.response import FullDetectorResponse, DetectorResponse
from cosipy.spacecraftfile import SpacecraftFile
from cosipy import test_data
from histpy import Histogram, HealpixAxis, Axis, Axes
import gc

# This is cosi.BinnedData
from cosipy import BinnedData, UnBinnedData
# this is for fitting and forward folding (#3ML is needed for spectral modeling)
from threeML import Model, Powerlaw
from astromodels import Band


# This is cosi_image_deconvolution
from cosipy.image_deconvolution import SpacecraftAttitudeExposureTable, CoordsysConversionMatrix, DataIF_COSI_DC2, ImageDeconvolution
from cosipy.util import fetch_wasabi_file
from cosipy.ts_map.TSMap import TSMap

# dependencies for data handling
import shutil
import os
from hashlib import md5

/home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/utils/file_utils.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


20:57:10 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=345273;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=910153;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/functions/functions_1D/functions.py#47\47]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=383539;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=574146;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/functions/functions_1D/functions.py#68\68]8;;\
                  will not be available.                                                                           

20:57:11 WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=410083;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=927662;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

         INFO      Starting 3ML!                                                                     ]8;id=171956;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=640569;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py#39\39]8;;\

...

         WARNING   Could not import plugin HAWCLike.py. Do you have the relative instrument         ]8;id=720509;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=282089;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py#144\144]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=905927;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=28310;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=330122;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=482525;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py#387\387]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=105454;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=67962;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py#387\387]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=883105;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=794404;file:///home/ricekong/miniforge3/envs/cosipy_env/lib/python3.10/site-packages/threeML/__init__.py#387\387]8;;\
                  performances in 3ML                                                                              

In [3]:
binned_bkg_scatt = Histogram.open("Data_reduction/Fe60_scatt_bkg_15sbin.hdf5")

binned_event_scatt = Histogram.open("Data_reduction/Fe60_scatt_event_15sbin.hdf5")

In [4]:
response = FullDetectorResponse.open("data/Response60FeHigh.o4.e1329_1336.s10201526728102.m1287.filtered.nonsparse.binnedimaging.imagingresponse_nside16.area.good_chunks.h5")

In [5]:
ccm = CoordsysConversionMatrix.open("Data_reduction/ccm_Fe60_scatt.hdf5")

In [11]:
ccm.binning_method

'ScAtt'

In [9]:
print(binned_bkg_scatt.axes.labels)
print(binned_bkg_scatt.contents.shape)

['ScAtt' 'Em' 'Phi' 'PsiChi']
(3096, 1, 60, 3072)


In [10]:
print(binned_event_scatt.axes.labels)
print(binned_event_scatt.contents.shape)

['ScAtt' 'Em' 'Phi' 'PsiChi']
(3096, 1, 60, 3072)


In [12]:
ccm.contents

Format,coo
Data Type,float64
Shape,"(3096, 3072, 3072)"
nnz,38043648
Density,0.0013020833333333333
Read-only,True
Size,1.1G
Storage ratio,0.01


In [6]:
%%time

data_interface = DataIF_COSI_DC2.load(name = "Fe60",
                                      event_binned_data = binned_event_scatt,
                                      dict_bkg_binned_data = {"Fe60 High": binned_bkg_scatt},
                                      rsp = response,
                                      coordsys_conv_matrix=ccm)

Loading the response matrix onto your computer memory...
Finished
... checking the axis ScAtt of the event and background files...
    --> pass (edges)
... checking the axis Em of the event and background files...
    --> pass (edges)
... checking the axis Phi of the event and background files...
    --> pass (edges)
... checking the axis PsiChi of the event and background files...
    --> pass (edges)
...checking the axis Em of the event and response files...
    --> pass (edges)
...checking the axis Phi of the event and response files...
    --> pass (edges)
...checking the axis PsiChi of the event and response files...
    --> pass (edges)
The axes in the event and background files are redefined. Now they are consistent with those of the response file.
Calculating an exposure map...
Finished...
CPU times: user 7.84 s, sys: 49.9 s, total: 57.7 s
Wall time: 1min 44s


In [8]:
parameter_filepath = "imagedeconvolution_Fe60.yml"

image_deconvolution = ImageDeconvolution()

image_deconvolution.set_dataset([data_interface])

image_deconvolution.read_parameterfile(parameter_filepath)

In [9]:
image_deconvolution.initialize()

#### Initialization Starts ####
<< Instantiating the model class AllSkyImage >>
---- parameters ----
coordinate: galactic
energy_edges:
  unit: keV
  value:
  - 1329.0
  - 1336.0
nside: 16
scheme: ring
unit: cm-2 s-1 sr-1

<< Setting initial values of the created model object >>
---- parameters ----
algorithm: flat
parameter:
  unit: cm-2 s-1 sr-1
  value:
  - 1e-4

<< Registering the deconvolution algorithm >>
---- parameters ----
algorithm: RLsimple
parameter:
  acceleration:
    activate: true
    alpha_max: 10.0
  background_normalization_optimization:
    activate: true
    range:
      Fe60 High:
      - 0.01
      - 10.0
  iteration_max: 1
  response_weighting:
    activate: true
    index: 0.5
  save_results:
    activate: false
    directory: ./results
    only_final_result: true
  smoothing: true
  smoothing_FWHM:
    unit: deg
    value: 0.5
  stopping_criteria:
    statistics: log-likelihood
    threshold: 0.01

#### Initialization Finished ####


In [ ]:
image_deconvolution.run_deconvolution()

#### Image Deconvolution Starts ####
<< Initialization >>


  0%|          | 0/1 [00:00<?, ?it/s]

## Iteration 1/1 ##
<< Pre-processing >>
<< E-step >>
<< M-step >>


In [ ]:
x, y = [], []

for result in image_deconvolution.results:
    x.append(result['iteration'])
    y.append(result['loglikelihood'])
    
plt.plot(x, y)
plt.grid()
plt.xlabel("iteration")
plt.ylabel("log-likelihood")
plt.show()

In [ ]:
x, y = [], []

for result in image_deconvolution.results:
    x.append(result['iteration'])
    y.append(result['alpha'])
    
plt.plot(x, y)
plt.grid()
plt.xlabel("iteration")
plt.ylabel("alpha")
plt.show()

In [ ]:
x, y = [], []

for result in image_deconvolution.results:
    x.append(result['iteration'])
    y.append(result['background_normalization']['Fe60 High'])
    
plt.plot(x, y)
plt.grid()
plt.xlabel("iteration")
plt.ylabel("background_normalization")
plt.show()

In [ ]:
def plot_reconstructed_image(result, source_position = None): # source_position should be (l,b) in degrees
    iteration = result['iteration']
    image = result['model']

    for energy_index in range(image.axes['Ei'].nbins):
        map_healpxmap = HealpixMap(data = image[:,energy_index], unit = image.unit)

        _, ax = map_healpxmap.plot('mollview')        
        
        _.colorbar.set_label(str(image.unit))
        
        if source_position is not None:
            ax.scatter(source_position[0]*u.deg, source_position[1]*u.deg, transform=ax.get_transform('world'), color = 'red')

        plt.title(label = f"iteration = {iteration}, energy_index = {energy_index} ({image.axes['Ei'].bounds[energy_index][0]}-{image.axes['Ei'].bounds[energy_index][1]})")

In [ ]:
iteration = 0


plot_reconstructed_image(image_deconvolution.results[iteration])

In [ ]:
iteration_idx = -1

result = image_deconvolution.results[iteration_idx]

iteration = result['iteration']
image = result['model']

data = image[:,0]
data[data <= 0 * data.unit] = 1e-12 * data.unit

hp.mollview(data, min = 2e-5, norm ='log', unit = str(data.unit), title = f'511 keV image at {iteration}th iteration', cmap = 'magma')

plt.show()